# Multi-Judging LLM System for Exam Answer Evaluation

This notebook implements the multi-judge exam evaluation architecture using **Google Flan-T5-Large** from HuggingFace — **no API key required**.

It runs entirely on Google Colab's free CPU/GPU. Enable GPU in Colab: `Runtime → Change runtime type → T4 GPU` for faster execution.

## 1. Install Dependencies

In [ ]:
!pip install -q transformers accelerate sentencepiece pandas scikit-learn seaborn matplotlib scipy

## 2. Load the Free LLM (Flan-T5-Large)

We use `google/flan-t5-large` — a free, instruction-following model by Google. Runs locally on Colab. **No API key needed.**

In [ ]:
import pandas as pd
import json
import re
import statistics
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, cohen_kappa_score
from scipy.stats import pearsonr
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ─── Configuration ───────────────────────────────────────────────
MODEL_NAME = "google/flan-t5-large"   # Free, no API key needed
SELF_CONSISTENCY_RUNS = 3
WEIGHTS = {"moderate": 0.50, "strict": 0.25, "lenient": 0.25}
MAX_NEW_TOKENS = 64

# ─── Load model once (no pipeline needed) ────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading {MODEL_NAME} on {'GPU' if device.type == 'cuda' else 'CPU'}...")
print("(First run will download ~3 GB — subsequent runs use the cache)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

print("Model loaded successfully!")

## 3. LLM Helper Function

In [ ]:
def call_model(prompt: str, retries: int = 3) -> str:
    """Call Flan-T5 directly using model.generate() — no pipeline, no API key."""
    for attempt in range(retries):
        try:
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=512
            ).to(device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,
                )
            return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        except Exception as e:
            print(f"  [Retry {attempt+1}] Error: {e}")
            time.sleep(2)
    return ""

## 4. Preprocessing Module

In [ ]:
def preprocess(question, model_answer, student_answer):
    """Standardizes input formatting."""
    return {
        "question": str(question).strip(),
        "model_answer": str(model_answer).strip(),
        "student_answer": str(student_answer).strip()
    }

## 5. Rubric Generator Module

Extracts key concepts and assigns marks from the reference answer.

> **Note:** Flan-T5 outputs plain text, not JSON. We parse it with regex and fall back to a default rubric if needed.

In [ ]:
def generate_rubric(data: dict, total_marks: int) -> list:
    """Generates a simplified rubric using the local model."""
    prompt = (
        f"You are an expert examiner. List the key concepts from the model answer below. "
        f"Assign marks to each concept so total marks = {total_marks}. "
        f"Format each line as: concept: marks\n\n"
        f"Question: {data['question'][:200]}\n"
        f"Model Answer: {data['model_answer'][:300]}"
    )
    raw = call_model(prompt)

    rubric = []
    for line in raw.strip().split("\n"):
        m = re.match(r"^(.+?):\s*(\d+)", line.strip())
        if m:
            rubric.append({"concept": m.group(1).strip(), "marks": int(m.group(2))})

    if not rubric:
        half = total_marks // 2 or 1
        rubric = [
            {"concept": "Core concept understanding", "marks": half},
            {"concept": "Explanation quality", "marks": total_marks - half}
        ]
    return rubric


def format_rubric(rubric_items: list) -> str:
    return "\n".join([f"  - {item['concept']} ({item['marks']} marks)" for item in rubric_items])

## 6. Multi-Judge Prompts & Evaluation

We use three distinct personas: **Strict**, **Moderate**, and **Lenient**.

> Each persona now has explicit score-range guidance so that Strict consistently scores lower and Lenient scores higher. A post-processing step enforces `strict ≤ moderate ≤ lenient` mathematically.

In [ ]:
# Each entry: (instruction, score_bias_hint)
# score_bias_hint tells the model exactly which part of the range to target.
PERSONA_CONFIG = {
    "strict": {
        "instruction": (
            "You are a STRICT examiner. "
            "You penalize harshly for ANY missing keyword, vague language, or incomplete explanation. "
            "Only award marks when the student's answer is precise and complete. "
            "When in doubt, DEDUCT marks. You are intentionally conservative."
        ),
        "bias_hint": "Give a LOW score (in the bottom third of the mark range). Be very tough."
    },
    "moderate": {
        "instruction": (
            "You are a MODERATE examiner. "
            "You allow reasonable paraphrasing and award marks when the core idea is correct. "
            "Balance strictness and leniency — award marks fairly."
        ),
        "bias_hint": "Give a FAIR score (in the middle of the mark range). Balance your judgment."
    },
    "lenient": {
        "instruction": (
            "You are a LENIENT examiner. "
            "You focus only on whether the student grasped the core idea. "
            "Award marks generously — even partial understanding should receive credit. "
            "When in doubt, AWARD marks. You are intentionally generous."
        ),
        "bias_hint": "Give a HIGH score (in the top third of the mark range). Be very generous."
    }
}


def build_judge_prompt(persona: str, data: dict, rubric_items: list, total_marks: int) -> str:
    cfg = PERSONA_CONFIG[persona]
    rubric_str = format_rubric(rubric_items)
    low  = round(total_marks * 0.10)
    mid  = round(total_marks * 0.50)
    high = round(total_marks * 0.80)
    range_hint = {
        "strict":   f"Typical strict scores: {low}–{mid} out of {total_marks}.",
        "moderate": f"Typical moderate scores: {mid-1}–{high} out of {total_marks}.",
        "lenient":  f"Typical lenient scores: {high}–{total_marks} out of {total_marks}."
    }[persona]

    return (
        f"{cfg['instruction']}\n\n"
        f"Scoring guideline: {cfg['bias_hint']}\n"
        f"{range_hint}\n\n"
        f"Rubric (Total: {total_marks} marks):\n{rubric_str}\n\n"
        f"Question: {data['question'][:150]}\n"
        f"Model Answer: {data['model_answer'][:200]}\n"
        f"Student Answer: {data['student_answer'][:200]}\n\n"
        f"Reply with ONLY a single integer between 0 and {total_marks}. No explanation."
    )


def run_single_judge(persona: str, data: dict, rubric: list, total_marks: int) -> dict:
    prompt = build_judge_prompt(persona, data, rubric, total_marks)
    raw = call_model(prompt)

    # Extract first number from response
    numbers = re.findall(r"\d+\.?\d*", raw)
    score = float(numbers[0]) if numbers else 0
    score = max(0, min(score, total_marks))

    return {
        "score": round(score, 2),
        "persona": persona,
        "covered_concepts": [],
        "missing_concepts": [],
        "feedback": f"Score assigned by {persona} judge: {score}/{total_marks}",
        "reasoning": raw
    }

## 7. Self-Consistency Engine

Runs each judge multiple times and averages the scores to reduce randomness.

In [ ]:
def run_judge_with_consistency(persona: str, data: dict, rubric: list, total_marks: int, runs: int = SELF_CONSISTENCY_RUNS) -> dict:
    scores, results = [], []

    for _ in range(runs):
        res = run_single_judge(persona, data, rubric, total_marks)
        scores.append(res["score"])
        results.append(res)

    avg_score = round(statistics.mean(scores), 2)
    std_score = round(statistics.stdev(scores) if len(scores) > 1 else 0.0, 2)
    best_run = min(results, key=lambda r: abs(r["score"] - avg_score))

    return {
        "persona": persona,
        "avg_score": avg_score,
        "std_dev": std_score,
        "covered_concepts": [],
        "missing_concepts": [],
        "feedback": best_run.get("feedback", ""),
        "reasoning": best_run.get("reasoning", "")
    }

## 8. Score Aggregation Module

> **Ordering enforcement:** After averaging, we apply a post-processing step to guarantee `strict ≤ moderate ≤ lenient`. This corrects any cases where the model fails to respect persona boundaries.

In [ ]:
def enforce_ordering(s, m, l):
    """
    Guarantee strict <= moderate <= lenient.
    Strategy: sort the three scores and re-assign them to
    (strict=lowest, moderate=middle, lenient=highest).
    """
    ordered = sorted([s, m, l])  # [lowest, middle, highest]
    return ordered[0], ordered[1], ordered[2]   # strict, moderate, lenient


def aggregate_scores(judge_results: dict, total_marks: int) -> dict:
    raw_s = judge_results["strict"]["avg_score"]
    raw_m = judge_results["moderate"]["avg_score"]
    raw_l = judge_results["lenient"]["avg_score"]

    # Enforce correct ordering: strict <= moderate <= lenient
    s, m, l = enforce_ordering(raw_s, raw_m, raw_l)

    final_score = round(WEIGHTS["moderate"] * m + WEIGHTS["strict"] * s + WEIGHTS["lenient"] * l, 2)
    final_score = max(0, min(final_score, total_marks))

    scores = [s, m, l]
    inter_var = round(statistics.variance(scores), 4) if len(scores) > 1 else 0.0
    disagreement = round(max(scores) - min(scores), 2)
    confidence = round((1 - disagreement / total_marks) * 100, 1) if total_marks > 0 else 100.0

    ordering_corrected = not (raw_s <= raw_m <= raw_l)

    return {
        "final_score": round(final_score),
        "final_score_float": final_score,
        "strict_score": s,
        "moderate_score": m,
        "lenient_score": l,
        "raw_strict": raw_s,
        "raw_moderate": raw_m,
        "raw_lenient": raw_l,
        "ordering_corrected": ordering_corrected,
        "inter_variance": inter_var,
        "disagreement": disagreement,
        "confidence_pct": confidence
    }

## 9. Feedback Synthesizer

In [ ]:
def generate_feedback(judge_results: dict, agg: dict, data: dict) -> dict:
    """Generate student-friendly feedback using the local model."""
    score = agg['final_score_float']
    total = data.get('total_marks', 10)

    prompt_strengths = (
        f"A student answered this exam question and scored {score}/{total}.\n"
        f"Question: {data['question'][:150]}\n"
        f"Student Answer: {data['student_answer'][:200]}\n"
        f"In one sentence, what did the student do well?"
    )
    strengths = call_model(prompt_strengths)

    prompt_missing = (
        f"A student answered this exam question and scored {score}/{total}.\n"
        f"Model Answer: {data['model_answer'][:200]}\n"
        f"Student Answer: {data['student_answer'][:200]}\n"
        f"In one sentence, what key point did the student miss?"
    )
    missing_points = call_model(prompt_missing)

    prompt_improve = (
        f"A student scored {score}/{total} on an exam question.\n"
        f"In one sentence, give one tip to improve their answer."
    )
    improvement = call_model(prompt_improve)

    return {
        "strengths": strengths or "Answer addressed some relevant points.",
        "missing_points": missing_points or "Some key concepts were not covered.",
        "improvement": improvement or "Review the model answer for missing details."
    }

## 10. Baseline Grader & Metrics

In [ ]:
def run_baseline(data: dict, total_marks: int) -> int:
    """Simple single-call baseline grader."""
    prompt = (
        f"You are an examiner. Score this answer from 0 to {total_marks}. Reply ONLY with a number.\n"
        f"Question: {data['question'][:150]}\n"
        f"Model Answer: {data['model_answer'][:200]}\n"
        f"Student Answer: {data['student_answer'][:200]}"
    )
    raw = call_model(prompt)
    numbers = re.findall(r"\d+\.?\d*", raw)
    try:
        return max(0, min(round(float(numbers[0])), total_marks))
    except:
        return 0


def compute_metrics(human, multi, base):
    def safe_p(a, b): return round(pearsonr(a, b)[0], 4) if np.std(a) > 0 and np.std(b) > 0 else 0.0
    def safe_q(a, b): return round(cohen_kappa_score(np.round(a).astype(int), np.round(b).astype(int), weights="quadratic"), 4)
    return {
        "multi_judge": {"MAE": round(mean_absolute_error(human, multi), 4), "Pearson": safe_p(human, multi), "QWK": safe_q(human, multi)},
        "baseline":    {"MAE": round(mean_absolute_error(human, base),  4), "Pearson": safe_p(human, base),  "QWK": safe_q(human, base)}
    }

## 11. Execution Pipeline over Dataset

Upload `mec.csv` to the Colab session before running this cell (drag-and-drop into the Files panel on the left).

In [ ]:
def process_dataset(df: pd.DataFrame, max_records: int = 5):
    records = df.head(max_records) if max_records else df
    results, human, multi, base = [], [], [], []

    for i, row in records.iterrows():
        print(f"\nProcessing [{i+1}/{len(records)}]...")
        total_m = int(row['total_marks'])
        data = preprocess(row['questions'], row['model_answer'], row['student_answer'])
        data['total_marks'] = total_m

        rubric = generate_rubric(data, total_m)
        print(f"  Rubric: {[r['concept'] for r in rubric]}")

        jr = {
            "strict":   run_judge_with_consistency("strict",   data, rubric, total_m),
            "moderate": run_judge_with_consistency("moderate", data, rubric, total_m),
            "lenient":  run_judge_with_consistency("lenient",  data, rubric, total_m)
        }
        agg = aggregate_scores(jr, total_m)
        fb  = generate_feedback(jr, agg, data)
        bs  = run_baseline(data, total_m)

        teacher_score = int(row['teacher_marks']) if pd.notna(row['teacher_marks']) else None

        if agg['ordering_corrected']:
            print(f"  ⚠️  Ordering corrected: raw (S={agg['raw_strict']}, M={agg['raw_moderate']}, L={agg['raw_lenient']}) "
                  f"→ enforced (S={agg['strict_score']}, M={agg['moderate_score']}, L={agg['lenient_score']})")

        results.append({
            "question":           data['question'],
            "student_answer":     data['student_answer'],
            "teacher_score":      teacher_score,
            "strict_score":       agg['strict_score'],
            "moderate_score":     agg['moderate_score'],
            "lenient_score":      agg['lenient_score'],
            "final_ai_score":     agg['final_score'],
            "baseline_score":     bs,
            "confidence":         agg['confidence_pct'],
            "disagreement":       agg['disagreement'],
            "ordering_corrected": agg['ordering_corrected'],
            "strengths":          fb['strengths'],
            "missing_points":     fb['missing_points'],
            "improvement":        fb['improvement']
        })

        if teacher_score is not None:
            human.append(teacher_score)
            multi.append(agg['final_score'])
            base.append(bs)

        print(f"  -> Human: {teacher_score} | "
              f"Strict: {agg['strict_score']} | Moderate: {agg['moderate_score']} | Lenient: {agg['lenient_score']} | "
              f"AI Final: {agg['final_score']} (Baseline: {bs}) | Confidence: {agg['confidence_pct']}%")

    df_res  = pd.DataFrame(results)
    metrics = compute_metrics(human, multi, base) if len(human) >= 2 else None
    return df_res, metrics


# ─── Load Dataset ──────────────────────────────────────────────────
# Option A: Upload manually to Colab (drag-and-drop to Files panel)
# Option B: Mount Google Drive and adjust the path below
print("Loading dataset...")
df = pd.read_csv("mec.csv")
print(f"Dataset loaded: {len(df)} records, columns: {list(df.columns)}")

# Run on first 5 records (increase max_records for more)
df_results, metrics = process_dataset(df, max_records=5)

print("\n=== Results ===")
print(df_results[["question", "teacher_score", "strict_score", "moderate_score",
                   "lenient_score", "final_ai_score", "baseline_score", "confidence"]].to_string())

print("\n=== Metrics ===")
print(metrics)

## 12. Score Comparison Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Multi-Judge Exam Evaluation Results", fontsize=14, fontweight="bold")

# Plot 1: Score Comparison
ax = axes[0]
idx = range(len(df_results))
ax.plot(idx, df_results["teacher_score"],  marker="o", label="Teacher Score",  linewidth=2)
ax.plot(idx, df_results["final_ai_score"], marker="s", label="AI Multi-Judge", linewidth=2)
ax.plot(idx, df_results["baseline_score"], marker="^", label="Baseline Score", linewidth=2, linestyle="--")
ax.set_title("Score Comparison per Question")
ax.set_xlabel("Record Index")
ax.set_ylabel("Score")
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Confidence Distribution
ax2 = axes[1]
sns.histplot(df_results["confidence"], kde=True, bins=10, ax=ax2, color="steelblue")
ax2.set_title("Confidence Score Distribution")
ax2.set_xlabel("Confidence (%)")
ax2.set_ylabel("Count")

plt.tight_layout()
plt.savefig("evaluation_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved as evaluation_results.png")

## 13. Judge Breakdown — Strict vs. Moderate vs. Lenient

This section shows the individual scores from each judge persona both as a **grouped bar chart** and as a **formatted table**.

> ✅ **Ordering guarantee:** Lenient score is always ≥ Moderate ≥ Strict. Rows where auto-correction was applied are flagged.

In [ ]:
# ── Grouped Bar Chart ─────────────────────────────────────────────
n = len(df_results)
x = np.arange(n)
width = 0.18

fig, ax = plt.subplots(figsize=(max(10, n * 1.8), 6))

bars_strict   = ax.bar(x - width * 1.5, df_results["strict_score"],   width, label="Strict",   color="#e74c3c", alpha=0.88)
bars_moderate = ax.bar(x - width * 0.5, df_results["moderate_score"], width, label="Moderate", color="#3498db", alpha=0.88)
bars_lenient  = ax.bar(x + width * 0.5, df_results["lenient_score"],  width, label="Lenient",  color="#2ecc71", alpha=0.88)
bars_final    = ax.bar(x + width * 1.5, df_results["final_ai_score"], width, label="Final AI", color="#9b59b6", alpha=0.88)

# Value labels
for bars in [bars_strict, bars_moderate, bars_lenient, bars_final]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.05, f"{h:.1f}",
                ha="center", va="bottom", fontsize=8, fontweight="bold")

# Teacher score overlay
if df_results["teacher_score"].notna().any():
    ax.plot(x, df_results["teacher_score"], marker="D", color="black",
            linestyle="--", linewidth=1.8, label="Teacher Score", zorder=5)

# Flag corrected rows
corrected_idx = df_results.index[df_results["ordering_corrected"]].tolist()
for ci in corrected_idx:
    ax.axvspan(ci - 0.45, ci + 0.45, alpha=0.08, color="orange", label="_auto-corrected")
    ax.text(ci, ax.get_ylim()[1] * 0.97, "⚠", ha="center", fontsize=12, color="orange")

ax.set_title("Judge Breakdown: Strict ≤ Moderate ≤ Lenient (enforced)", fontsize=13, fontweight="bold")
ax.set_xlabel("Record Index")
ax.set_ylabel("Score")
ax.set_xticks(x)
ax.set_xticklabels([f"Q{i+1}" for i in range(n)])
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("judge_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved as judge_breakdown.png")

# ── Formatted & Styled Table ──────────────────────────────────────
print("\n" + "=" * 75)
print("  JUDGE BREAKDOWN TABLE  (Strict ≤ Moderate ≤ Lenient — guaranteed)")
print("=" * 75)

table_df = df_results[[
    "question", "teacher_score",
    "strict_score", "moderate_score", "lenient_score",
    "final_ai_score", "confidence", "ordering_corrected"
]].copy()
table_df.index = [f"Q{i+1}" for i in range(len(table_df))]
table_df.columns = ["Question", "Teacher", "Strict", "Moderate", "Lenient",
                    "Final AI", "Conf %", "Auto-Corrected?"]
table_df["Question"] = table_df["Question"].str[:35] + "..."

try:
    from IPython.display import display
    display(
        table_df.style
        .format({"Strict": "{:.2f}", "Moderate": "{:.2f}", "Lenient": "{:.2f}",
                 "Final AI": "{:.2f}", "Conf %": "{:.1f}"})
        .background_gradient(subset=["Strict", "Moderate", "Lenient", "Final AI"], cmap="RdYlGn")
        .apply(lambda col: ["background-color: #ffe0b2" if v else "" for v in col],
               subset=["Auto-Corrected?"])
        .set_caption("Multi-Judge Score Breakdown — Strict ≤ Moderate ≤ Lenient")
    )
except Exception:
    print(table_df.to_string())

n_corrected = df_results["ordering_corrected"].sum()
print(f"\n📊 {n_corrected}/{n} records had their ordering auto-corrected (⚠ rows).")
print("   This happens when Flan-T5 ignores persona bias — enforced ordering ensures correctness.")